# EDA — Spain 2027 EV Charging Network
## IE Sustainability Datathon 2026 · Iberdrola Challenge

Full exploratory data analysis across all mandatory datasets (M1–M3) and grid capacity.

| Section | Dataset | Key question |
|---|---|---|
| 1 | M1 Road Network | Which roads and how long? |
| 2 | M2 Charging Baseline | What infrastructure exists nationally? |
| 3 | M2 Interurban & Coverage Gap | How well covered are interurban roads? |
| 4 | M3 BEV Fleet Projection | How many EVs by 2027? |
| 5 | Cross-Dataset Synthesis | What is the 2027 supply–demand gap? |
| 6 | Grid Capacity (Endesa) | Can the grid support new HPC hubs? |
| 7 | National Grid (REE) | What is EV load as % of national demand? |
| 8 | Summary | All key KPIs in one place |

---
## 0. Setup

In [ ]:
import os, json, warnings, glob
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import requests
from pyproj import Transformer
warnings.filterwarnings('ignore')

# ── Colab: clone repo if needed ───────────────────────────────────────────
IN_COLAB = 'google.colab' in __import__('sys').modules
if IN_COLAB:
    import subprocess
    if not os.path.isdir('IberdrolaDatathon'):
        subprocess.run(['git','clone','https://github.com/NOSIEMPRE/IberdrolaDatathon.git'], check=True)
    os.chdir('IberdrolaDatathon/Datathon')

# ── Base path detection ───────────────────────────────────────────────────
for _d in ['.', '..']:
    if os.path.isdir(os.path.join(_d, 'Data')):
        BASE = os.path.abspath(_d); break
else:
    raise FileNotFoundError('Cannot find Data/ folder')

RTIG_PATH     = f'{BASE}/Data/raw/road_routes_spain/carreteras_RTIG.geojson'
M2_ALL        = f'{BASE}/Data/interim/m2_charging_sites_all.csv'
M2_INTERURBAN = f'{BASE}/Data/interim/m2_charging_sites_interurban.csv'
M2_COVERAGE   = f'{BASE}/Data/interim/m2_road_coverage.csv'
M3_PARQUET    = f'{BASE}/Data/raw/ev_fleet_projections_datosgob/parquet'
M3_JSON       = f'{BASE}/Data/processed/m3_ev_projection.json'
ENDESA_CSV    = f'{BASE}/Data/external/grid_capacity_endesa/endesa_demanda_2026_03.csv'

print(f'BASE = {BASE}')
for label, path in [
    ('RTIG geojson',      RTIG_PATH),
    ('M2 all sites',      M2_ALL),
    ('M2 interurban',     M2_INTERURBAN),
    ('M2 road coverage',  M2_COVERAGE),
    ('M3 parquets',       M3_PARQUET),
    ('M3 projection JSON',M3_JSON),
    ('Endesa CSV',        ENDESA_CSV),
]:
    exists = os.path.isdir(path) if 'parquet' in label else os.path.isfile(path)
    print(f'  {"OK" if exists else "MISSING":7s} {label}')

---
## 1. Road Network — RTIG Interurban Roads (M1)

In [ ]:
roads = gpd.read_file(RTIG_PATH)

total_km = roads['length_m'].sum() / 1000
print(f'Segments   : {len(roads):,}')
print(f'Total km   : {total_km:,.0f} km')
print()
print('Road type breakdown:')
vc = roads['Tipo_de_via'].value_counts()
km = roads.groupby('Tipo_de_via')['length_m'].sum().div(1000).round(0).astype(int)
for t in vc.index:
    print(f'  {t:<35s}  {vc[t]:>4} segs  {km[t]:>6,} km')

In [ ]:
TYPE_COLORS = {
    'Autopista libre\\Autovía': '#1565C0',
    'Autopista peaje':          '#0288D1',
    'Carretera nacional':       '#2E7D32',
    'Otros':                    '#9E9E9E',
}
default_color = '#9E9E9E'

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
for tipo in roads['Tipo_de_via'].dropna().unique():
    color = TYPE_COLORS.get(tipo, default_color)
    roads[roads['Tipo_de_via'] == tipo].plot(ax=ax, color=color, linewidth=0.8, alpha=0.9)
handles = [mpatches.Patch(color=TYPE_COLORS.get(t, default_color), label=t)
           for t in roads['Tipo_de_via'].dropna().unique()]
ax.legend(handles=handles, fontsize=8, loc='lower left')
ax.set_title(f'RTIG Interurban Road Network ({len(roads):,} segments, {total_km:,.0f} km)')

ax = axes[1]
km_by_type = roads.groupby('Tipo_de_via')['length_m'].sum().div(1000).sort_values()
colors = [TYPE_COLORS.get(t, default_color) for t in km_by_type.index]
km_by_type.plot(kind='barh', ax=ax, color=colors)
ax.set_xlabel('km')
ax.set_title('Network Length by Road Type')
for i, v in enumerate(km_by_type):
    ax.text(v + 50, i, f'{v:,.0f} km', va='center', fontsize=8)

plt.tight_layout(); plt.show()

**Findings — Road Network Baseline**

Spain's eligible interurban network spans **~29,050 km** across 1,535 road segments. Autopistas/autovías dominate (~19,000 km), while carreteras nacionales add ~8,700 km. This network is the spatial backbone for all subsequent charging-site placement analysis.

---
## 2. EV Charging Sites — National Baseline (M2)

In [ ]:
# Load all national sites if available, else fall back to interurban
if os.path.isfile(M2_ALL):
    gdf_all = gpd.GeoDataFrame(
        pd.read_csv(M2_ALL),
        geometry=gpd.points_from_xy(
            pd.read_csv(M2_ALL)['longitude'],
            pd.read_csv(M2_ALL)['latitude']),
        crs='EPSG:4326')
    print(f'Loaded national dataset: {len(gdf_all):,} sites')
else:
    gdf_all = gpd.GeoDataFrame(
        pd.read_csv(M2_INTERURBAN),
        geometry=gpd.points_from_xy(
            pd.read_csv(M2_INTERURBAN)['longitude'],
            pd.read_csv(M2_INTERURBAN)['latitude']),
        crs='EPSG:4326')
    print(f'National CSV not found — using interurban only: {len(gdf_all):,} sites')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Power distribution
ax = axes[0, 0]
bins   = [0, 7.4, 22, 50, 100, 150, 250, 350, 1000]
labels = ['≤7.4 (AC slow)', '7.4–22 (AC)', '22–50 (AC fast)',
          '50–100 (DC)', '100–150 (DC fast)', '150–250 (HPC)',
          '250–350 (HPC+)', '>350 (Ultra)']
gdf_all['power_band'] = pd.cut(gdf_all['max_power_kw'], bins=bins, labels=labels)
gdf_all['power_band'].value_counts().sort_index().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Sites by Max Power Band')
ax.set_xlabel('Number of sites')

# 2. Top 15 operators
ax = axes[0, 1]
gdf_all['operator'].value_counts().head(15)[::-1].plot(kind='barh', ax=ax, color='darkorange')
ax.set_title('Top 15 Operators by Site Count')
ax.set_xlabel('Number of sites')

# 3. Connectors per site
ax = axes[1, 0]
gdf_all['n_refill_points'].clip(upper=20).value_counts().sort_index().plot(
    kind='bar', ax=ax, color='forestgreen')
ax.set_title('Charging Connectors per Site (capped at 20)')
ax.set_xlabel('Connectors'); ax.set_ylabel('Sites')

# 4. Geographic distribution
ax = axes[1, 1]
gdf_all.plot(ax=ax, markersize=1, color='crimson', alpha=0.4)
ax.set_title(f'All EV Charging Sites in Spain (n={len(gdf_all):,})')

plt.tight_layout(); plt.show()

print(f'Sites with DC fast (≥50 kW) : {(gdf_all["max_power_kw"] >= 50).sum():,}')
print(f'Sites with HPC (≥150 kW)    : {(gdf_all["max_power_kw"] >= 150).sum():,}')
print(f'Avg connectors per site     : {gdf_all["n_refill_points"].mean():.1f}')

**Findings — Infrastructure Quality Gap**

Spain has **12,074 registered public charging sites**, but the power distribution reveals a critical maturity issue: **~56% of sites are sub-50 kW** (AC-only, unsuitable for interurban travel). Only ~8% meet HPC threshold (≥150 kW). The market is fragmented across 300+ operators, with the top 5 controlling ~40% of sites.

---
## 3. Interurban Filter & Road Coverage Gap (M2)

In [ ]:
df_interurban = pd.read_csv(M2_INTERURBAN)
gdf_interurban = gpd.GeoDataFrame(
    df_interurban,
    geometry=gpd.points_from_xy(df_interurban['longitude'], df_interurban['latitude']),
    crs='EPSG:4326'
)
n_interurban = len(gdf_interurban)
n_total      = len(gdf_all)

print(f'Total national sites : {n_total:,}')
print(f'Interurban sites     : {n_interurban:,}  ({n_interurban/n_total*100:.1f}%)')
print(f'Unique roads covered : {gdf_interurban["Carretera"].nunique():,}')
print()
print('Power tier breakdown (interurban):')
power_bins   = [0, 50, 150, float('inf')]
power_labels = ['<50 kW (AC/DC slow)', '50–150 kW (DC fast)', '≥150 kW (HPC)']
power_colors = ['gold', 'darkorange', 'crimson']
gdf_interurban['power_cat'] = pd.cut(
    gdf_interurban['max_power_kw'], bins=power_bins, labels=power_labels)
tier_counts = gdf_interurban['power_cat'].value_counts().reindex(power_labels)
n_stations  = n_interurban
for label, count in tier_counts.items():
    print(f'  {label:<25s}: {count:,} ({count/n_stations*100:.0f}%)')

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
gdf_all.plot(ax=ax, markersize=1, color='lightgrey', alpha=0.5, label=f'All ({n_total:,})')
gdf_interurban.plot(ax=ax, markersize=3, color='steelblue', alpha=0.8, label=f'Interurban ({n_interurban:,})')
roads.plot(ax=ax, linewidth=0.3, color='darkorange', alpha=0.4, label='RTIG roads')
ax.legend(fontsize=8); ax.set_title('All vs Interurban Charging Sites')

ax = axes[1]
roads.plot(ax=ax, linewidth=0.3, color='lightgrey', alpha=0.5)
handles = []
for label, color in zip(power_labels, power_colors):
    subset = gdf_interurban[gdf_interurban['power_cat'] == label]
    subset.plot(ax=ax, markersize=4, color=color, alpha=0.85)
    handles.append(mpatches.Patch(color=color, label=f'{label} ({len(subset):,})'))
ax.legend(handles=handles, fontsize=8, loc='lower left')
ax.set_title('Interurban Sites by Power Tier')

plt.tight_layout(); plt.show()

In [ ]:
coverage = pd.read_csv(M2_COVERAGE)
GAP_KM = 50
total_km_net = coverage['length_m'].sum() / 1000
gap_km_val   = coverage[coverage['has_gap']]['length_m'].sum() / 1000
covered_pct  = (1 - gap_km_val / total_km_net) * 100

print(f'Total network  : {total_km_net:,.0f} km')
print(f'Covered (≤{GAP_KM}km): {total_km_net-gap_km_val:,.0f} km  ({covered_pct:.1f}%)')
print(f'Gap (>{GAP_KM}km)    : {gap_km_val:,.0f} km  ({100-covered_pct:.1f}%)')
print()
print('Top roads with largest gap:')
top_gaps = (coverage.groupby('Carretera')['nearest_station_m']
            .max().sort_values(ascending=False).head(10).div(1000).round(1))
for road, km_gap in top_gaps.items():
    print(f'  {road:<12s}  {km_gap:.1f} km')

fig, axes = plt.subplots(2, 1, figsize=(14, 16))

# Map
ax = axes[0]
roads_cov = roads[['Carretera','geometry']].copy()
roads_cov = roads_cov.merge(coverage[['Carretera','has_gap']].drop_duplicates('Carretera'), on='Carretera', how='left')
roads_cov[~roads_cov['has_gap'].fillna(False)].plot(ax=ax, color='steelblue', linewidth=0.8, label=f'Covered (≤{GAP_KM} km)')
roads_cov[roads_cov['has_gap'].fillna(False)].plot(ax=ax, color='crimson', linewidth=2.0, label=f'Gap (>{GAP_KM} km)')
gdf_interurban.plot(ax=ax, markersize=3, color='gold', zorder=5, label='Existing stations')
ax.legend(fontsize=9); ax.set_title(f'Road Coverage Gap Map ({GAP_KM} km threshold)')

# Histogram
ax = axes[1]
coverage['nearest_km'] = coverage['nearest_station_m'] / 1000
coverage['nearest_km'].clip(upper=200).hist(bins=40, ax=ax, color='steelblue', edgecolor='white')
ax.axvline(GAP_KM, color='crimson', linestyle='--', linewidth=1.5, label=f'{GAP_KM} km threshold')
ax.set_xlabel('Distance to nearest station (km)')
ax.set_ylabel('Road segments')
ax.set_title('Distribution: Distance to Nearest Existing Station')
ax.legend()

plt.tight_layout(); plt.show()

**Findings — Coverage Gap**

**99.6% of Spain's interurban road network is within 50 km of an existing station**, so the challenge is not geographic coverage — it's **quality**. The N-502, N-122, and a handful of carreteras nacionales remain true white spots. For M4 placement, the priority should be upgrading existing low-power stops to HPC hubs, not filling geographic gaps.

---
## 4. BEV Fleet Projection 2027 (M3)

**Model: Logistic market penetration curve** fitted on Spain's annual BEV registration data.

Why not SARIMA? The SARIMA model (tested on the DGT parquets) predicted *declining* new BEV sales post-2024 due to noise and seasonal artefacts in the monthly data. A logistic penetration model is more appropriate for EV adoption, which follows an S-curve driven by falling costs and infrastructure build-out.

In [ ]:
from scipy.optimize import curve_fit

parquet_files = sorted(glob.glob(f'{M3_PARQUET}/*.parquet'))

if parquet_files:
    dfs = [pd.read_parquet(f) for f in parquet_files]
    df_raw = pd.concat(dfs, ignore_index=True)
    df_raw['COD_TIPO']           = df_raw['COD_TIPO'].astype(str).str.strip()
    df_raw['CLAVE_TRAMITE']      = df_raw['CLAVE_TRAMITE'].astype(str).str.strip()
    df_raw['COD_PROPULSION_ITV'] = df_raw['COD_PROPULSION_ITV'].astype(str).str.strip()
    df_cars = df_raw[
        (df_raw['COD_TIPO'] == '40') &
        (df_raw['CLAVE_TRAMITE'].isin(['1','5','B']))
    ].copy()
    df_cars['year'] = df_cars['FEC_MATRICULA'].astype(str).str[4:8].astype(int, errors='ignore')
    df_cars = df_cars[df_cars['year'].between(2015, 2023)]
    annual = pd.DataFrame({
        'total': df_cars.groupby('year').size(),
        'bev':   df_cars[df_cars['COD_PROPULSION_ITV']=='6'].groupby('year').size(),
    }).fillna(0).astype(int)
    annual['penetration'] = annual['bev'] / annual['total']
    DATA_SOURCE = 'DGT parquets'
else:
    # Reliable published values (ANFAC / DGT official stats)
    annual = pd.DataFrame({
        'total': {2019:960000, 2020:851000, 2021:860000, 2022:900000, 2023:950000},
        'bev':   {2019:2516,   2020:3625,   2021:9601,   2022:15882,  2023:27928},
    })
    annual['penetration'] = annual['bev'] / annual['total']
    DATA_SOURCE = 'hardcoded (no parquets)'

print(f'Source: {DATA_SOURCE}')
print()
print(f'{"Year":>6}  {"Total cars":>12}  {"BEV":>8}  {"Penetration":>12}')
for yr, row in annual.iterrows():
    print(f'{yr:>6}  {row["total"]:>12,}  {row["bev"]:>8,}  {row["penetration"]:>11.2%}')

In [ ]:
def logistic(t, L, k, t0):
    return L / (1 + np.exp(-k * (t - t0)))

# Fit on 2019-2023 (post-policy-push; excludes noisy pre-2019 data)
fit_data = annual[annual.index >= 2019]
years_fit = fit_data.index.astype(float).values
pens_fit  = fit_data['penetration'].values

popt, _ = curve_fit(
    logistic, years_fit, pens_fit,
    p0=[0.30, 0.5, 2027],
    bounds=([0.10, 0.05, 2023], [1.0, 5.0, 2035]),
    maxfev=10000
)
L, k, t0 = popt
print(f'Logistic params:  L={L:.3f}  k={k:.3f}  t0={t0:.1f}')
print(f'Implied max penetration: {L*100:.1f}%')
print(f'Inflection year:         {t0:.1f}')

# Plot: observed penetration + fitted curve
t_smooth = np.linspace(2015, 2032, 200)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(annual.index, annual['penetration']*100, color='steelblue', zorder=5, label='Observed')
ax.plot(t_smooth, logistic(t_smooth, *popt)*100, color='darkorange', lw=2, label='Logistic fit')
ax.axvline(2027, color='grey', ls=':', lw=1)
ax.scatter([2027], [logistic(2027,*popt)*100], color='red', s=80, zorder=6,
           label=f'2027: {logistic(2027,*popt)*100:.1f}%')
ax.set_xlabel('Year'); ax.set_ylabel('BEV market share (%)')
ax.set_title('BEV Market Penetration — Logistic Fit')
ax.legend()

ax = axes[1]
proj_years = range(2024, 2028)
proj_pens  = {y: logistic(float(y), *popt) for y in proj_years}
proj_pens_df = pd.Series(proj_pens)
all_pens = pd.concat([annual['penetration'], proj_pens_df])
colors_pen = ['steelblue']*len(annual) + ['darkorange']*len(proj_years)
ax.bar(all_pens.index, all_pens.values*100, color=colors_pen)
ax.set_xlabel('Year'); ax.set_ylabel('BEV market share (%)')
ax.set_title('Annual BEV Penetration Rate (observed + projected)')
ax.legend(handles=[
    mpatches.Patch(color='steelblue',  label='Historical'),
    mpatches.Patch(color='darkorange', label='Projected')])

plt.tight_layout(); plt.show()

In [ ]:
# Project annual new BEV registrations 2024-2027
base_market = int(annual['total'].iloc[-1])   # use 2023 market size as base
proj_bev = {
    y: int(base_market * (1.02 ** (y - 2023)) * logistic(float(y), *popt))
    for y in range(2024, 2028)
}
fleet_baseline_2023    = int(annual['bev'].sum())
projected_2024_2027    = sum(proj_bev.values())
total_ev_projected_2027 = fleet_baseline_2023 + projected_2024_2027

print(f'Fleet baseline (all BEV 2015–2023): {fleet_baseline_2023:>10,}')
print(f'Projected new registrations 2024–2027: {projected_2024_2027:>8,}')
print(f'total_ev_projected_2027:             {total_ev_projected_2027:>10,}')
print()
for y, n in proj_bev.items():
    print(f'  {y}: {n:,} new BEVs  ({logistic(float(y),*popt)*100:.1f}% market share)')

# Visualise
hist_bev = annual['bev']
proj_s   = pd.Series(proj_bev)
all_annual = pd.concat([hist_bev, proj_s])
cumulative = all_annual.cumsum()
n_hist = len(hist_bev)
bar_colors = ['steelblue']*n_hist + ['darkorange']*(len(all_annual)-n_hist)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(all_annual.index, all_annual.values, color=bar_colors)
axes[0].set_title('Annual BEV Registrations, Spain')
axes[0].set_ylabel('New registrations')
axes[0].legend(handles=[
    mpatches.Patch(color='steelblue',  label='Historical'),
    mpatches.Patch(color='darkorange', label='Projected')])

axes[1].plot(cumulative.index, cumulative.values, marker='o', color='steelblue')
axes[1].axvline(x=2023.5, color='grey', ls=':', lw=1)
axes[1].scatter([2027], [total_ev_projected_2027], color='red', s=100, zorder=5,
                label=f'2027 Fleet: {total_ev_projected_2027:,}')
axes[1].set_title('Cumulative BEV Fleet, Spain (projected to 2027)')
axes[1].set_ylabel('Total vehicles')
axes[1].legend()

plt.tight_layout(); plt.show()

# Save for downstream notebooks
from datetime import date
os.makedirs(f'{BASE}/Data/processed', exist_ok=True)
with open(M3_JSON, 'w') as f:
    json.dump({
        'total_ev_projected_2027': total_ev_projected_2027,
        'fleet_baseline_2023': fleet_baseline_2023,
        'projected_registrations_2024_2027': projected_2024_2027,
        'logistic_params': {'L': round(float(L),4), 'k': round(float(k),4), 't0': round(float(t0),2)},
        'annual_forecast': {str(y): v for y, v in proj_bev.items()},
        'model': 'Logistic penetration curve fitted on 2019-2023',
        'generated_at': str(date.today()),
    }, f, indent=2)
print(f'Saved → {M3_JSON}')

**Findings — BEV Fleet Projection (Logistic Model)**

The logistic penetration curve captures Spain's S-shaped EV adoption trajectory. BEV market share is projected to reach **~8–12% by 2027**, up from 2.9% in 2023. Total cumulative BEV fleet by end-2027: **~300,000–400,000 vehicles** depending on market size assumptions. This monotonically increasing forecast is more economically credible than the SARIMA alternative, which predicted declining annual sales due to seasonal noise in the training data.

---
## 5. Cross-Dataset Synthesis — 2027 Supply–Demand Gap

In [ ]:
benchmark_low  = 20    # EVs per station at comfortable utilisation
benchmark_high = 50    # EVs per station at high utilisation
ev_per_station = total_ev_projected_2027 / n_stations

stations_needed_low  = int(total_ev_projected_2027 / benchmark_low)
stations_needed_high = int(total_ev_projected_2027 / benchmark_high)
gap_low  = max(0, stations_needed_low  - n_stations)
gap_high = max(0, stations_needed_high - n_stations)

print(f'Existing interurban stations : {n_stations:,}')
print(f'Projected EVs (2027)         : {total_ev_projected_2027:,}')
print(f'EVs per existing station     : {ev_per_station:,.0f}')
print()
print(f'Benchmark (20 EVs/station)   : {stations_needed_low:,} stations needed  →  gap = {gap_low:,}')
print(f'Benchmark (50 EVs/station)   : {stations_needed_high:,} stations needed  →  gap = {gap_high:,}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Station gap bar
ax = axes[0]
ax.barh(['Existing', 'Needed (20/st)', 'Needed (50/st)'],
        [n_stations, stations_needed_low, stations_needed_high],
        color=['steelblue', 'crimson', 'darkorange'])
ax.set_xlabel('Interurban stations')
ax.set_title('2027: Existing vs Required Stations')

# Road map coloured by power tier
ax = axes[1]
roads.plot(ax=ax, color='lightgrey', linewidth=0.4, alpha=0.6)
for label, color in zip(power_labels, power_colors):
    gdf_interurban[gdf_interurban['power_cat'] == label].plot(
        ax=ax, markersize=3, color=color, alpha=0.8)
handles2 = [mpatches.Patch(color=c, label=l) for l, c in zip(power_labels, power_colors)]
ax.legend(handles=handles2, fontsize=8, loc='lower left')
ax.set_title('Interurban Stations — Power Tier Map')

plt.tight_layout(); plt.show()

**Findings — 2027 Gap**

Spain will need **~4,400–11,000 additional interurban stations** by 2027 depending on utilisation. More urgently, **84% of existing stations are below 50 kW**, so the primary investment priority is upgrading existing sites to HPC (≥150 kW), not just adding more sites.

---
## 6. Grid Capacity — Endesa Distribution Zone

**Source:** Endesa `endesa_demanda_2026_03.csv` — firm available capacity (MW) by substation. Coverage: Andalucía, Cataluña, Aragón, Extremadura, Canarias, Illes Balears (~6 CCAAs).

In [ ]:
df_raw = pd.read_csv(ENDESA_CSV, sep=';', decimal=',', encoding='utf-8-sig')
df_raw = df_raw.rename(columns={
    'Coordenada UTM X': 'utm_x', 'Coordenada UTM Y': 'utm_y',
    'Nivel de Tensión (kV)': 'kv',
    'Capacidad firme disponible (MW)': 'available_mw',
    'Nombre Subestación': 'nombre', 'Comunidad Autónoma': 'ccaa',
    'Subestación': 'sub_id', 'Provincia.1': 'provincia',
})
for col in ['utm_x','utm_y','kv','available_mw']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# One row per substation: max available capacity across voltage levels
sub = (df_raw.groupby('sub_id').agg(
    utm_x=('utm_x','first'), utm_y=('utm_y','first'),
    nombre=('nombre','first'), ccaa=('ccaa','first'),
    max_kv=('kv','max'), available_mw=('available_mw','max'),
).reset_index().dropna(subset=['utm_x','utm_y']))

# UTM ETRS89 Zone 30N → WGS84
t = Transformer.from_crs('EPSG:25830','EPSG:4326', always_xy=True)
sub['longitude'], sub['latitude'] = t.transform(sub['utm_x'].values, sub['utm_y'].values)
valid = sub['longitude'].between(-10,5) & sub['latitude'].between(35,44)
gdf_sub = gpd.GeoDataFrame(
    sub[valid].reset_index(drop=True),
    geometry=gpd.points_from_xy(sub[valid]['longitude'], sub[valid]['latitude']),
    crs='EPSG:4326'
)

# Grid status: 1 HPC = 0.15 MW; 10-charger hub = 1.5 MW
def classify(mw):
    if mw <= 0:    return 'Congested'
    elif mw < 1.5: return 'Moderate'
    else:          return 'Sufficient'
gdf_sub['grid_status'] = gdf_sub['available_mw'].apply(classify)

status_counts = gdf_sub['grid_status'].value_counts()
print(f'Substations (mainland): {len(gdf_sub):,}')
print(f'CCAAs covered          : {gdf_sub["ccaa"].nunique()}')
for status, n in status_counts.items():
    print(f'  {status:<12s}: {n:,} ({n/len(gdf_sub)*100:.0f}%)')
print(f'Zero-capacity subs     : {(gdf_sub["available_mw"]==0).sum():,} ({(gdf_sub["available_mw"]==0).mean()*100:.0f}%)')

In [ ]:
STATUS_COLORS = {'Congested':'#D32F2F','Moderate':'#F57C00','Sufficient':'#388E3C'}

fig, axes = plt.subplots(1, 3, figsize=(21, 7))

# Map
ax = axes[0]
roads.plot(ax=ax, color='#E0E0E0', linewidth=0.4)
for status, color in STATUS_COLORS.items():
    gdf_sub[gdf_sub['grid_status']==status].plot(ax=ax, markersize=4, color=color, alpha=0.8)
handles_s = [mpatches.Patch(color=c, label=f'{s} ({status_counts.get(s,0):,})')
             for s, c in STATUS_COLORS.items()]
ax.legend(handles=handles_s, fontsize=8)
ax.set_title('Endesa Substation Grid Status')

# Status distribution
ax = axes[1]
colors_bar = [STATUS_COLORS[s] for s in status_counts.index]
status_counts.plot(kind='bar', ax=ax, color=colors_bar, rot=0)
ax.set_title('Grid Status Distribution')
ax.set_ylabel('Substations')

# Available capacity distribution (excluding congested)
ax = axes[2]
gdf_sub[gdf_sub['available_mw']>0]['available_mw'].clip(upper=50).hist(
    bins=30, ax=ax, color='#388E3C', edgecolor='white')
ax.axvline(1.5, color='red', ls='--', lw=1.5, label='Hub threshold (1.5 MW)')
ax.set_xlabel('Available capacity (MW)')
ax.set_ylabel('Substations')
ax.set_title('Available Capacity (non-zero substations, capped 50 MW)')
ax.legend()

plt.tight_layout(); plt.show()

### 6b. Grid Status per Interurban Station

In [ ]:
gdf_sub_m  = gdf_sub.to_crs('EPSG:25830')
gdf_sta_m  = gdf_interurban.to_crs('EPSG:25830')

nearest = gpd.sjoin_nearest(
    gdf_sta_m,
    gdf_sub_m[['sub_id','available_mw','grid_status','geometry']],
    how='left', distance_col='dist_to_sub_m'
)
nearest = nearest[~nearest.index.duplicated(keep='first')]
nearest['in_endesa_zone'] = nearest['dist_to_sub_m'] <= 25_000
nearest['assigned_status'] = nearest.apply(
    lambda r: r['grid_status'] if r['in_endesa_zone'] else 'i-DE/Viesgo (no data)', axis=1)

ALL_COLORS = {**STATUS_COLORS, 'i-DE/Viesgo (no data)': '#9E9E9E'}
status_assigned = nearest['assigned_status'].value_counts()
print('Grid status assigned to interurban stations:')
for status, n in status_assigned.items():
    print(f'  {status:<28s}: {n:,} ({n/len(nearest)*100:.0f}%)')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax = axes[0]
roads.plot(ax=ax, color='#E0E0E0', linewidth=0.3)
for status, color in ALL_COLORS.items():
    subset = nearest[nearest['assigned_status']==status]
    if len(subset):
        gpd.GeoDataFrame(subset, geometry='geometry', crs='EPSG:25830').to_crs('EPSG:4326').plot(
            ax=ax, markersize=4, color=color, alpha=0.8)
handles_a = [mpatches.Patch(color=c, label=f'{s} ({status_assigned.get(s,0):,})')
             for s, c in ALL_COLORS.items() if s in status_assigned]
ax.legend(handles=handles_a, fontsize=8, loc='lower left')
ax.set_title('Interurban Stations — Assigned Grid Status')

ax = axes[1]
colors_bar2 = [ALL_COLORS.get(s,'#9E9E9E') for s in status_assigned.index]
status_assigned.plot(kind='bar', ax=ax, color=colors_bar2, rot=30)
ax.set_title('Grid Status Distribution at Interurban Stations')
ax.set_ylabel('Stations')

plt.tight_layout(); plt.show()

**Findings — Grid Capacity**

~65% of Endesa substations already have **zero available firm capacity**, concentrated on high-traffic corridors (A-4, A-7). For new HPC hubs in Endesa territory, grid reinforcement works must be co-planned. Stations in i-DE/Viesgo territory (~60% of the network) require separate capacity assessment — flagged as 'no data' for M4.

---
## 7. National Grid Context (REE)

**Source:** REE `apidatos.ree.es` — public API, no authentication required.

In [ ]:
REE = 'https://apidatos.ree.es'

def ree_get(endpoint, **params):
    try:
        r = requests.get(f'{REE}{endpoint}', params=params, timeout=15)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f'REE API unavailable: {e}')
        return None

data = ree_get('/es/datos/demanda/evolucion',
               start_date='2023-01-01', end_date='2023-12-31',
               time_trunc='year', geo_limit='peninsular')

if data:
    included = data.get('included', [])
    demand_twh = None
    for item in included:
        if item.get('type') == 'Demanda en b.c.':
            vals = item['attributes']['values']
            demand_twh = vals[0]['value'] / 1e9 if vals else None
            break
    if demand_twh is None:
        demand_twh = 250  # 2023 actual ~250 TWh
    print(f'Spain peninsular demand 2023: {demand_twh:.1f} TWh')
else:
    demand_twh = 250
    print('Using fallback: 250 TWh (2023 actual)')

# EV annual energy demand estimate
# Average consumption: ~15 kWh/100km, ~15,000 km/year → 2,250 kWh/EV/year
ev_kwh_per_year = 2250
ev_twh = total_ev_projected_2027 * ev_kwh_per_year / 1e9
share_pct = ev_twh / demand_twh * 100

print(f'Projected EVs (2027)         : {total_ev_projected_2027:,}')
print(f'Est. EV annual demand        : {ev_twh:.2f} TWh')
print(f'As % of national demand      : {share_pct:.2f}%')

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(['EV demand (2027)', 'Other demand (2027 est.)'],
        [ev_twh, demand_twh - ev_twh],
        color=['darkorange','steelblue'])
ax.set_xlabel('TWh')
ax.set_title(f'EV Load as Share of National Demand ({share_pct:.2f}%)')
plt.tight_layout(); plt.show()

**Findings — National Grid Impact**

Spain's total electricity demand is ~250 TWh/year. All 220,000+ projected EVs would add only ~**0.5 TWh** (< 0.2% of national demand). The grid constraint is **local distribution** (substations), not national generation capacity. This validates the Endesa substation analysis as the binding constraint for M4.

---
## 8. Key Findings Summary

In [ ]:
rows = [
    ('M1  Eligible road network',          f'{total_km:,.0f} km',           '1,535 segments'),
    ('M2  National charging sites',        f'{n_total:,}',                  'Pre-filter'),
    ('M2  Interurban sites (≤500m)',        f'{n_stations:,}',               f'{n_stations/n_total*100:.0f}% of national'),
    ('M2  HPC sites (≥150 kW)',            f'{(gdf_interurban["max_power_kw"]>=150).sum():,}', f'{(gdf_interurban["max_power_kw"]>=150).mean()*100:.0f}% of interurban'),
    ('M2  Road coverage',                  '99.6%',                         '≤50 km to nearest station'),
    ('M3  BEV fleet 2027 (SARIMA)',        f'{total_ev_projected_2027:,}',  'Central estimate'),
    ('M3  EVs per existing station 2027',  f'{ev_per_station:,.0f}',        ''),
    ('Gap Additional stations needed',     f'{gap_low:,} – {gap_high:,}',   '20–50 EVs/station benchmark'),
    ('R2  Congested substations (Endesa)', f'{status_counts.get("Congested",0):,} / {len(gdf_sub):,}', f'{status_counts.get("Congested",0)/len(gdf_sub)*100:.0f}%'),
    ('REE EV load as % national demand',  f'{share_pct:.2f}%',             'Local, not national constraint'),
]
print(f'{"KPI":<45} {"Value":<22} {"Note"}')
print('-' * 90)
for kpi, val, note in rows:
    print(f'{kpi:<45} {val:<22} {note}')

### Outputs for the Modelling Team (M4)

| File | Contents |
|---|---|
| `Data/interim/m2_charging_sites_interurban.csv` | 3,679 interurban stations with road assignment |
| `Data/interim/m2_road_coverage.csv` | Distance to nearest station per road segment |
| `Data/processed/m3_ev_projection.json` | SARIMA fleet projection + annual forecast |
| `Data/external/grid_capacity_endesa/` | Substation-level grid capacity (Endesa zone) |